# Task 1.4 — Switching Hybrid Recommender

We implemented a switching hybrid that selects between Popular and SLIMElastic based on the number of training interactions per user. Users with interaction counts below or equal to a threshold receive popularity-based recommendations, while the remaining users receive recommendations from SLIMElastic.

The threshold was optimized on the validation set using MRR@10. Thresholds between −1 and 10 achieved the same maximum validation MRR@10 of 0.4165. Increasing the threshold to 20 and 50 reduced MRR@10 to 0.3937 and 0.3417, respectively. We selected −1 as the first threshold achieving the maximum validation score.

On the test set, the switching hybrid achieved MRR@10 of 0.5113 and NDCG@10 of 0.3108, identical to SLIMElastic. This is expected because the selected threshold routes all users to the personalized recommender.

These results indicate that switching to the popularity-based recommender did not improve recommendation accuracy in our experimental setting. The additional switching mechanism therefore provided no benefit over SLIMElastic.

## 1. Setup


In [31]:
import numpy as np
import pandas as pd
import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import get_model, init_seed

TOP_K = 10
config = Config(model="SLIMElastic", dataset="ml-100k",
                config_file_list=["../recbole/config/SLIMElastic/ml-100k.yaml"])
init_seed(config["seed"], config["reproducibility"])
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)
model = get_model("SLIMElastic")(config, train_data.dataset).to(config["device"])
model.eval()

# Training-only user and item interaction frequencies (RecBole internal IDs).
train_inter = train_data.dataset.inter_feat
user_field = config["USER_ID_FIELD"]
item_field = config["ITEM_ID_FIELD"]
user_counts = torch.bincount(train_inter[user_field].long(), minlength=dataset.user_num)
item_counts = torch.bincount(train_inter[item_field].long(), minlength=dataset.item_num)
print("Users:", dataset.user_num, "Items:", dataset.item_num)
print("Training profile lengths:", user_counts[1:].float().quantile(torch.tensor([0., .25, .5, .75, 1.])))

Users: 944 Items: 1683
Training profile lengths: tensor([ 16.,  27.,  53., 120., 591.])


## 2. Switching rule and evaluation

The popularity score of an item is its interaction count in the training set. The two models are not blended: for each user, one complete score vector is selected. Previously seen items and the padding item are excluded from ranking.

In [32]:
def metrics_at_k(recommended, relevant, k=10):
    relevant = set(relevant)
    hits = np.array([item in relevant for item in recommended[:k]], dtype=float)
    discounts = 1 / np.log2(np.arange(2, k + 2))
    ideal = discounts[:min(len(relevant), k)].sum()
    positions = np.flatnonzero(hits)
    return {
        f"Recall@{k}": hits.sum() / len(relevant) if relevant else 0.,
        f"MRR@{k}": 1. / (positions[0] + 1) if len(positions) else 0.,
        f"NDCG@{k}": (hits * discounts).sum() / ideal if ideal else 0.,
        f"Hit@{k}": float(hits.any()),
        f"Precision@{k}": hits.sum() / k,
    }

def evaluate_switch(eval_data, threshold, k=TOP_K):
    results = []
    with torch.no_grad():
        for interaction, history_index, positive_u, positive_i in eval_data:
            interaction = interaction.to(config["device"])
            personalized = model.full_sort_predict(interaction).view(len(interaction), -1)
            # Same popularity scores for every user; one vector per batch row.
            popularity = item_counts.to(personalized.device, dtype=personalized.dtype)
            popularity = popularity.unsqueeze(0).expand_as(personalized)

            # RecBole full-sort batches index users by their row position.
            user_ids = interaction[user_field].long().cpu()
            sparse = (user_counts[user_ids] <= threshold).to(personalized.device)
            scores = torch.where(sparse[:, None], popularity, personalized).clone()
            scores[:, 0] = -torch.inf
            if history_index is not None:
                history_u, history_i = history_index
                scores[history_u.to(scores.device), history_i.to(scores.device)] = -torch.inf

            top_items = torch.topk(scores, k=k, dim=1).indices.cpu()
            positives = {}
            for u, i in zip(positive_u.tolist(), positive_i.tolist()):
                positives.setdefault(u, set()).add(i)
            for row_idx, relevant in positives.items():
                results.append(metrics_at_k(top_items[row_idx].tolist(), relevant, k))

    if not results:
        raise ValueError("No evaluable users in the supplied split")
    return {key: float(np.mean([r[key] for r in results])) for key in results[0]}

## 3. Select the threshold on validation data

The threshold $-1$ means *always SLIMElastic*, while a threshold at least as large as the maximum profile length means *always popularity*. The intermediate thresholds implement switching.

In [33]:
max_count = int(user_counts.max().item())
thresholds = sorted(set([-1, 0, 2, 5, 10, 20, 50, max_count]))
validation_rows = []
for threshold in thresholds:
    result = evaluate_switch(valid_data, threshold)
    validation_rows.append({"threshold": threshold, **result})


validation_df = pd.DataFrame(validation_rows)
display(validation_df.sort_values("MRR@10", ascending=False))
best_threshold = int(validation_df.loc[validation_df["MRR@10"].idxmax(), "threshold"])
print("Selected validation threshold:", best_threshold)

,threshold,Recall@10,MRR@10,NDCG@10,Hit@10,Precision@10
0,-1,0.227123,0.416487,0.249512,0.762460,0.170944
1,0,0.227123,0.416487,0.249512,0.762460,0.170944
2,2,0.227123,0.416487,0.249512,0.762460,0.170944
3,5,0.227123,0.416487,0.249512,0.762460,0.170944
4,10,0.227123,0.416487,0.249512,0.762460,0.170944
5,20,0.208035,0.393650,0.232354,0.733828,0.167126
6,50,0.160474,0.341678,0.193257,0.656416,0.151538
7,591,0.116644,0.231435,0.122114,0.550371,0.089396


Selected validation threshold: -1


In [36]:
max_count = int(user_counts.max().item())
thresholds = sorted(set([-1, 0, 2, 5, 10, 20, 50, max_count]))

# Users evaluated in the validation split
validation_users = set()

for interaction, _, _, _ in valid_data:
    validation_users.update(
        interaction[user_field].long().cpu().tolist()
    )

validation_users = sorted(validation_users)

validation_rows = []

for threshold in thresholds:
    result = evaluate_switch(valid_data, threshold)

    n_popular = sum(
        user_counts[u].item() <= threshold
        for u in validation_users
    )

    validation_rows.append({
        "threshold": threshold,
        "Users using Popular": n_popular,
        "Users using SLIMElastic": len(validation_users) - n_popular,
        **result
    })

validation_df = pd.DataFrame(validation_rows)

display(validation_df[
    [
        "threshold",
        "Users using Popular",
        "Users using SLIMElastic",
        "MRR@10",
        "NDCG@10"
    ]
])

best_threshold = int(
    validation_df.loc[
        validation_df["MRR@10"].idxmax(),
        "threshold"
    ]
)

print("Selected validation threshold:", best_threshold)

,threshold,Users using Popular,Users using SLIMElastic,MRR@10,NDCG@10
0,-1,0,943,0.416487,0.249512
1,0,0,943,0.416487,0.249512
2,2,0,943,0.416487,0.249512
3,5,0,943,0.416487,0.249512
4,10,0,943,0.416487,0.249512
5,20,121,822,0.393650,0.232354
6,50,458,485,0.341678,0.193257
7,591,943,0,0.231435,0.122114


Selected validation threshold: -1


## 4. Final test evaluation

Apply the selected threshold **without changing it** after inspecting test results. The two endpoint configurations serve as component baselines.

In [34]:
test_rows = []
for name, threshold in [
    ("Popularity", max_count),
    ("SLIMElastic", -1),
    ("Switching hybrid", best_threshold),
]:
    test_rows.append({"Model": name, **evaluate_switch(test_data, threshold)})
test_results = pd.DataFrame(test_rows).set_index("Model")
display(test_results.round(4))
print("Compare these results with the weighted hybrid reported in Task 1.3.")

,Recall@10,MRR@10,NDCG@10,Hit@10,Precision@10
Model,,,,,
Popularity,0.1278,0.2673,0.1420,0.5769,0.1032
SLIMElastic,0.2646,0.5113,0.3108,0.8038,0.2071
Switching hybrid,0.2646,0.5113,0.3108,0.8038,0.2071


Compare these results with the weighted hybrid reported in Task 1.3.


In [35]:
print("Best threshold:", best_threshold)

display(
    validation_df[
        ["threshold", "MRR@10", "NDCG@10"]
    ].sort_values("threshold")
)

print("Users routed to Popular:",
      int((user_counts[1:] <= best_threshold).sum()))

print("Users routed to SLIMElastic:",
      int((user_counts[1:] > best_threshold).sum()))

Best threshold: -1


,threshold,MRR@10,NDCG@10
0,-1,0.416487,0.249512
1,0,0.416487,0.249512
2,2,0.416487,0.249512
3,5,0.416487,0.249512
4,10,0.416487,0.249512
5,20,0.393650,0.232354
6,50,0.341678,0.193257
7,591,0.231435,0.122114


Users routed to Popular: 0
Users routed to SLIMElastic: 943


In [ ]:
for threshold in [-1, 0, 2, 5, 10, 20, 50, 591]:
    n_pop = sum(
        user_counts[u] <= threshold
        for u in validation_users
    )

    print(
        f"Threshold {threshold:>3}: "
        f"{n_pop} users routed to Popular"
    )

# Task 1.5 

In [37]:
thresholds = [
    -1, 0, 5, 10, 12, 14, 16,
    18, 20, 25, 30, 40, 50
]

switch_results = []

for threshold in thresholds:
    metrics = evaluate_switch(
        valid_data,
        threshold
    )

    switch_results.append({
        "threshold": threshold,
        **metrics
    })

switch_df = pd.DataFrame(switch_results)

display(switch_df)

best_threshold = int(
    switch_df.loc[
        switch_df["MRR@10"].idxmax(),
        "threshold"
    ]
)

print("Best threshold:", best_threshold)

,threshold,Recall@10,MRR@10,NDCG@10,Hit@10,Precision@10
0,-1,0.227123,0.416487,0.249512,0.762460,0.170944
1,0,0.227123,0.416487,0.249512,0.762460,0.170944
2,5,0.227123,0.416487,0.249512,0.762460,0.170944
3,10,0.227123,0.416487,0.249512,0.762460,0.170944
4,12,0.227123,0.416487,0.249512,0.762460,0.170944
5,14,0.227123,0.416487,0.249512,0.762460,0.170944
6,16,0.223942,0.410956,0.245827,0.757158,0.170308
7,18,0.216518,0.401776,0.238876,0.746554,0.168823
8,20,0.208035,0.393650,0.232354,0.733828,0.167126
9,25,0.189300,0.379285,0.218880,0.710498,0.163097


Best threshold: -1


In [38]:
tuned_switch_test = evaluate_switch(
    test_data,
    best_threshold
)

pd.Series(tuned_switch_test)

Recall@10       0.264639
MRR@10          0.511346
NDCG@10         0.310771
Hit@10          0.803818
Precision@10    0.207105
dtype: float64